In [92]:
from qldpc_sim import *
from qldpc_sim.qldpc_experiment import *
from qldpc_sim.qec_code import *
from qldpc_sim.data_structure import *
from qldpc_sim.ckbb_surgery import *
import stim
import numpy as np

In [93]:
def setup_cnot_exp():
    patch1 = RotatedSurfaceCode.from_distance(3, code_name="ancilla")
    patch2 = RotatedSurfaceCode.from_distance(3, code_name="control")
    patch3 = RotatedSurfaceCode.from_distance(3, code_name="target")
    qm = qldpc_experiment.QuantumMemory(size=600)
    patches = [patch1, patch2, patch3]
    mapqb = {}
    for c in patches:
        for lq in c.logical_qubits:
            mapqb[lq.logical_x] = c
            mapqb[lq.logical_z] = c

    ctx = Context(
        codes=[patch1, patch2, patch3], logical_qubits=patch1.logical_qubits + patch2.logical_qubits + patch3.logical_qubits, initial_assignement=mapqb, memory=qm
    )

    return ctx

In [94]:
context = setup_cnot_exp()

ancilla = context.codes[0]
control = context.codes[1]
target = context.codes[2]

cnot_program = (
    [
        InitializeCode(
            code=control,
            context=context,
            tag=f"init_{control.id}",
            initial_state=PauliEigenState.Z_plus,
        )
    ]
    + [
        InitializeCode(
            code=target,
            context=context,
            tag=f"init_{target.id}",
            initial_state=PauliEigenState.Z_plus,
        )
    ]
    + [
        InitializeCode(
            code=ancilla,
            context=context,
            tag=f"init_{ancilla.id}",
            initial_state=PauliEigenState.X_plus,
        )
    ]
    + [
        StabMeasurement(code=p, context=context, tag=f"isb_2_{p.id}", round=3)
        for p in [control, ancilla, target]
    ]
    + [
        CKBBMeasurement(
            distance=3,
            context=context,
            tag="Merge_C-A",
            logical_targets=[
                control.logical_qubits[0].logical_z,
                ancilla.logical_qubits[0].logical_z,
            ],
        ),
    ]
    + [
        StabMeasurement(code=p, context=context, tag=f"isb_2_{p.id}", round=3)
        for p in [control, ancilla, target]
    ]
    + [
        CKBBMeasurement(
            distance=3,
            context=context,
            tag="Merge_T-A",
            logical_targets=[
                ancilla.logical_qubits[0].logical_x,
                target.logical_qubits[0].logical_x,
            ],
        ),
    ]
    + [
        StabMeasurement(code=p, context=context, tag=f"isb_3_{p.id}", round=3)
        for p in [control, ancilla, target]
    ]
    + [
        LM(
            logical_targets=[ancilla.logical_qubits[0].logical_z],
            context=context,
            tag=f"final_ancilla",
            basis=PauliChar.Z,
        ),
        LM(
            logical_targets=[target.logical_qubits[0].logical_z],
            context=context,
            tag=f"final_target",
            basis=PauliChar.Z,
        ),
        LM(
            logical_targets=[control.logical_qubits[0].logical_z],
            context=context,
            tag=f"final_control",
            basis=PauliChar.Z,
        ),
    ]
)

In [95]:
from typing import Dict, List, Set, Tuple

from qldpc_sim.qldpc_experiment.qec_gadget import LogicGadget


def _format_nodes(nodes):
    return [
        getattr(node, "tag", str(node))
        for node in sorted(nodes, key=lambda n: getattr(n, "tag", str(n)))
    ]


def _logical_qubit_name(logical_qubit: LogicalQubit) -> str:
    return getattr(logical_qubit, "name", str(logical_qubit))


def _describe_operator(op: LogicalOperator) -> str:
    return (
        f"type={op.logical_type}, "
        f"id={getattr(op, 'id', None)}, "
        f"nodes={_format_nodes(getattr(op, 'target_nodes', ())) }"
    )


def _print_frame_state(ctx, header: str):
    print(f"\n=== {header} ===")
    if not ctx.frame_tracker.frame_corrections:
        print("Frame tracker is empty.")
        return

    for logical_qubit, correction in sorted(
        ctx.frame_tracker.frame_corrections.items(),
        key=lambda item: _logical_qubit_name(item[0]),
    ):
        print(
            f"{_logical_qubit_name(logical_qubit)}: "
            f"X={_format_nodes(correction.correction_X_cond)} "
            f"Z={_format_nodes(correction.correction_Z_cond)}"
        )


def evaluate(
    ctx,
    program: List[QECGadget],
    verbose: bool = True,
) -> Tuple[List[str], Dict[str, Set[int]], MeasurementRecord]:
    """Evaluate a qLDPC program in the context."""
    logical_outcomes: Dict[str, Set[int]] = {}
    instructions = []
    if verbose:
        _print_frame_state(ctx, "initial frame state")

    for gadget_index, gadget in enumerate(program, start=1):
        if verbose:
            print(f"\n{'#' * 80}")
            print(f"Gadget {gadget_index}: {gadget.tag} ({type(gadget).__name__})")

        compilers, outcomes = gadget.build_compiler_instructions()
        if verbose:
            print(f"  compilers={len(compilers)}, outcomes={len(outcomes)}")

        for compiler_index, compiler in enumerate(compilers, start=1):
            n_instructions, measurements = compiler.compile(ctx.memory)
            instructions.extend(n_instructions)

            if measurements is not None:
                ctx.record.add_measurements(measurements)

            if verbose:
                n_meas = 0 if measurements is None else len(measurements.measured_nodes)
                print(
                    f"  compiler[{compiler_index}] {type(compiler).__name__}: "
                    f"instr={len(n_instructions)}, measured_nodes={n_meas}"
                )

        for outcome_index, outcome in enumerate(outcomes, start=1):
            if verbose:
                print(
                    f"  outcome[{outcome_index}] {outcome.tag}: "
                    f"type={outcome.type}, size={outcome.size}, "
                    f"nodes={_format_nodes(outcome.measured_nodes)}"
                )
                if isinstance(outcome.target, LogicalOperator):
                    print(f"    target operator: {_describe_operator(outcome.target)}")
                else:
                    print(f"    target: {outcome.target}")

            ctx.record.add_event(outcome)

            if outcome.type == EventType.FRAME_CORRECTION:
                logical_qubit = ctx.map_operator_to_qubits[outcome.target]
                before = set(
                    ctx.frame_tracker.get_correction(
                        logical_qubit, outcome.target.logical_type
                    )
                )
                ctx.frame_tracker.add_correction(
                    logical_qubit, outcome.target.logical_type, outcome.measured_nodes
                )
                after = set(
                    ctx.frame_tracker.get_correction(
                        logical_qubit, outcome.target.logical_type
                    )
                )

                if verbose:
                    print(
                        f"    frame({_logical_qubit_name(logical_qubit)}, {outcome.target.logical_type}): "
                        f"before={_format_nodes(before)} -> after={_format_nodes(after)}"
                    )

            if outcome.type == EventType.OBSERVABLE:
                # Always create an entry for observable events, including set-target observables.
                if outcome.tag not in logical_outcomes:
                    logical_outcomes[outcome.tag] = set()

                direct_nodes = set(outcome.measured_nodes)
                logical_outcomes[outcome.tag] |= direct_nodes

                target_ops = []
                if isinstance(outcome.target, LogicalOperator):
                    target_ops = [outcome.target]
                elif isinstance(outcome.target, set):
                    target_ops = [
                        op for op in outcome.target if isinstance(op, LogicalOperator)
                    ]

                merged_nodes = set()
                for target_op in target_ops:
                    logical_qubit = ctx.map_operator_to_qubits[target_op]
                    tracked_correction = set(
                        ctx.frame_tracker.get_correction(
                            logical_qubit,
                            target_op.logical_type,
                        )
                    )
                    merged_nodes |= tracked_correction

                logical_outcomes[outcome.tag] |= merged_nodes

                if verbose:
                    print(f"    observable direct nodes: {_format_nodes(direct_nodes)}")
                    if target_ops:
                        print(f"    observable merged frame nodes: {_format_nodes(merged_nodes)}")
                    print(
                        f"    observable final nodes: {_format_nodes(logical_outcomes[outcome.tag])}"
                    )

        if verbose:
            _print_frame_state(ctx, f"frame after gadget {gadget.tag}")

    if verbose:
        print(f"\n{'#' * 80}")
        _print_frame_state(ctx, "final frame state")
    return instructions, logical_outcomes, ctx.record

In [96]:
from collections import Counter
from qldpc_sim.qldpc_experiment.interpreter import concat_events_per_sample, xor_event_nodes

ctx = context
ctx.record = MeasurementRecord()
ctx.frame_tracker = FrameState()
ctx.memory = QuantumMemory(size=600)
prog = cnot_program

print("Reset context state before debug run:")
print(f"  record measurements: {len(ctx.record.measurements)}")
print(f"  record events: {len(ctx.record.events)}")
print(f"  frame entries: {len(ctx.frame_tracker.frame_corrections)}")

p, lo, rec = evaluate(ctx, prog, verbose=True)

print("\nRecorded events:")
for event, value in rec.events.items():
    print(f"  {event.tag}: value={value}, nodes={[node.tag for node in event.measured_nodes]}")

print("\nRecorded measurements and outcomes:")
for n, m in rec.measurements.items():
    print(f"{n.tag} - {m}")

print("\nEvent-to-measurement indices:")
event_idx_map = None
try:
    event_idx_map = rec.event_to_measurement_idx
except ValueError as err:
    print(f"Warning: could not build event_to_measurement_idx ({err})")

if event_idx_map is not None:
    for event, included_idx in event_idx_map.items():
        print(f"  {event.tag}: {sorted(included_idx)}")

print("\nFinal frame tracker snapshot:")
for logical_qubit, correction in sorted(
    ctx.frame_tracker.frame_corrections.items(),
    key=lambda item: getattr(item[0], "name", str(item[0])),
):
    print(
        f"  {getattr(logical_qubit, 'name', logical_qubit)}: "
        f"X={_format_nodes(correction.correction_X_cond)} "
        f"Z={_format_nodes(correction.correction_Z_cond)}"
    )

print("\nLogical outcomes:")
for key, value in lo.items():
    print(f"  {key}: {sorted(node.tag for node in value)}")

Reset context state before debug run:
  record measurements: 0
  record events: 0
  frame entries: 0

=== initial frame state ===
Frame tracker is empty.

################################################################################
Gadget 1: init_a5c1f4d8-6500-4c86-910e-b291a1b779a1 (InitializeCode)
  compilers=1, outcomes=0
  compiler[1] StabilisersMeasurementCompiler: instr=58, measured_nodes=8

=== frame after gadget init_a5c1f4d8-6500-4c86-910e-b291a1b779a1 ===
Frame tracker is empty.

################################################################################
Gadget 2: init_d1465861-a932-4ea5-baf9-46fcabe793a0 (InitializeCode)
  compilers=1, outcomes=0
  compiler[1] StabilisersMeasurementCompiler: instr=58, measured_nodes=8

=== frame after gadget init_d1465861-a932-4ea5-baf9-46fcabe793a0 ===
Frame tracker is empty.

################################################################################
Gadget 3: init_64ae4ce7-a798-4552-8c76-12b8952a36e6 (InitializeCode)
  compi

In [97]:
from qldpc_sim.qldpc_experiment.record import MeasurementRecord
from qldpc_sim.qldpc_experiment.pauli_frame import FrameState

# Re-evaluate program from a clean state to rebuild instructions and references.
ctx = context
ctx.record = MeasurementRecord()
ctx.frame_tracker = FrameState()
ctx.memory = QuantumMemory(size=600)

instructions, logical_outcomes, rec = evaluate(ctx, cnot_program, verbose=False)

# Build a noiseless circuit by dropping explicit noise instructions.
noise_ops = {
    "X_ERROR",
    "Y_ERROR",
    "Z_ERROR",
    "DEPOLARIZE1",
    "DEPOLARIZE2",
    "PAULI_CHANNEL_1",
    "PAULI_CHANNEL_2",
    "CORRELATED_ERROR",
    "ELSE_CORRELATED_ERROR",
    "E",
}

noiseless_instructions = []
for inst in instructions:
    line = str(inst).strip()
    op = line.split()[0] if line else ""
    if op in noise_ops:
        continue
    noiseless_instructions.append(line)

circuit = stim.Circuit("\n".join(noiseless_instructions))
sample = circuit.compile_sampler().sample(shots=1)[0]

# Observable event boundaries: each node maps to its latest measurement index before event end.
observable_end_idx = {}
for event, end_idx in rec.events.items():
    if event.type == EventType.OBSERVABLE:
        observable_end_idx[event.tag] = end_idx

observable_indices = {}
observable_bits = {}
for obs_tag, nodes in logical_outcomes.items():
    end_idx = observable_end_idx.get(obs_tag, rec.num_measurement_recorded)
    idx = set()
    for node in nodes:
        valid_idx = [i for i in rec.measurements.get(node, []) if i < end_idx]
        if valid_idx:
            idx.add(valid_idx[-1])

    idx = sorted(idx)
    observable_indices[obs_tag] = idx

    bit = 0
    for i in idx:
        bit ^= int(sample[i])
    observable_bits[obs_tag] = bit

print(f"Noiseless instruction count: {len(noiseless_instructions)}")
print(f"Measurement count: {len(sample)}")

print("\nObservable XOR sources (measurement indices):")
for k in sorted(observable_indices):
    print(f"  {k}: {observable_indices[k]}")

print("\nFinal logical outcomes from XOR:")
for k in sorted(observable_bits):
    print(f"  {k}: {observable_bits[k]}")

# Also show event-level XORs to mirror event_to_measurement_idx references.
event_bits = {}
for event, idx_set in rec.event_to_measurement_idx.items():
    bit = 0
    for i in sorted(idx_set):
        bit ^= int(sample[i])
    event_bits[event.tag] = (sorted(idx_set), bit)

print("\nEvent XOR outcomes (from event_to_measurement_idx):")
for tag in sorted(event_bits):
    idx, bit = event_bits[tag]
    print(f"  {tag}: idx={idx} -> {bit}")

Noiseless instruction count: 1482
Measurement count: 567

Observable XOR sources (measurement indices):
  LogicalMeasurement_final_ancilla: [465, 481, 482, 558, 559, 560]
  LogicalMeasurement_final_control: [564, 565, 566]
  LogicalMeasurement_final_target: [561, 562, 563]
  Merge_C-A_parity_outcome: [188, 189, 191, 192, 193, 194, 195, 200, 201, 203, 205, 208, 210, 212, 214, 218, 219, 222]
  Merge_T-A_parity_outcome: [416, 419, 421, 422, 423, 427, 431, 439, 441, 442, 444, 447, 449, 451, 452, 454, 457, 458]

Final logical outcomes from XOR:
  LogicalMeasurement_final_ancilla: 0
  LogicalMeasurement_final_control: 0
  LogicalMeasurement_final_target: 1
  Merge_C-A_parity_outcome: 1
  Merge_T-A_parity_outcome: 1

Event XOR outcomes (from event_to_measurement_idx):
  LogicalMeasurement_final_ancilla: idx=[558, 559, 560] -> 0
  LogicalMeasurement_final_control: idx=[564, 565, 566] -> 0
  LogicalMeasurement_final_target: idx=[561, 562, 563] -> 1
  Merge_C-A_parity_outcome: idx=[188, 189, 191

In [99]:
# Final post-processing over multiple shots.
# Rule (mod-2): target_corrected = target_raw XOR ancilla_final XOR zz_parity.

ancilla_tag = "LogicalMeasurement_final_ancilla"
target_tag = "LogicalMeasurement_final_target"
zz_parity_tag = "Merge_C-A_parity_outcome"

missing = [
    tag
    for tag in [ancilla_tag, target_tag, zz_parity_tag]
    if tag not in observable_indices
]
if missing:
    raise KeyError(f"Missing observable indices required for post-processing: {missing}")

shots = 5
multi_sample = circuit.compile_sampler().sample(shots=shots)


def xor_from_indices(sample_row, idx_list):
    bit = 0
    for i in idx_list:
        bit ^= int(sample_row[i])
    return bit


print(f"Post-processed logical outcomes over {shots} shots:")
print("shot | ancilla_final | target_raw | zz_parity | target_corrected")
print("-----+---------------+------------+-----------+-----------------")

for shot_idx, row in enumerate(multi_sample, start=1):
    ancilla_final = xor_from_indices(row, observable_indices[ancilla_tag])
    target_raw = xor_from_indices(row, observable_indices[target_tag])
    zz_parity = xor_from_indices(row, observable_indices[zz_parity_tag])

    target_corrected = target_raw ^ ancilla_final ^ zz_parity

    print(
        f"{shot_idx:>4} | {ancilla_final:>13} | {target_raw:>10} | {zz_parity:>9} | {target_corrected:>16}"
    )

Post-processed logical outcomes over 5 shots:
shot | ancilla_final | target_raw | zz_parity | target_corrected
-----+---------------+------------+-----------+-----------------
   1 |             1 |          1 |         0 |                0
   2 |             1 |          1 |         0 |                0
   3 |             0 |          0 |         0 |                0
   4 |             1 |          0 |         1 |                0
   5 |             0 |          0 |         0 |                0
